# 06 Journalism Search And Retrieval Evaluation

Milestone 6 turns the structured vocabulary into grounded table discovery. The engine builds one search document per table, indexes lexical text with SQLite FTS5/BM25, indexes semantic table descriptions with PEARL-small and FAISS CPU, and compares title-only BM25, all-vocabulary BM25, PEARL semantic retrieval, and fused StatVocab retrieval on STAR questions.

Search results identify relevant source tables only. They do not answer numeric questions, and every returned result includes score components plus source evidence.

In [ ]:
!statvocab build-search-index --config configs/core.yaml

In [ ]:
!statvocab evaluate --config configs/evaluation.yaml --area retrieval

In [ ]:
import json
from pathlib import Path

current_index = Path('outputs/search/current_index.json')
if current_index.exists():
    current = json.loads(current_index.read_text())
    manifest = json.loads(Path(current['index_manifest']).read_text())
    index_summary = json.loads(Path(manifest['summary']).read_text())
else:
    index_summary = {'status': 'not yet built'}
index_summary

In [ ]:
metrics_path = Path('report/retrieval_metrics.json')
metrics = json.loads(metrics_path.read_text()) if metrics_path.exists() else {'status': 'not yet evaluated'}
metrics

In [ ]:
if 'systems' in metrics:
    {
        system: {
            question_set: {
                key: values[key]
                for key in ('HitRate@10', 'Relevance@5', 'MRR', 'p95_latency_ms')
            }
            for question_set, values in system_metrics.items()
        }
        for system, system_metrics in metrics['systems'].items()
    }
else:
    metrics

The validation split is used only to choose among fixed score-fusion presets. Final-test metrics remain separated in `report/retrieval_metrics.json`, and detailed ranked predictions are written under `outputs/retrieval/<run_id>/retrieval_predictions.csv`.